# PINN Robustness and Hard-Boundary Tests

Five-seed robustness analysis of the standard soft-boundary PINN and
hard-boundary PINN experiments for Pe = 100 and 1000.


In [ ]:
import os
import csv
import sys
import time
import random
import platform
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


# 1. Settings

PE_CASES = [100.0, 1000.0]
SEEDS = [1234, 2345, 3456, 4567, 5678]
HARD_BC_SEED = 1234
OUTPUT_DIR = "pinn_robustness_hardbc_results"

WIDTH = 64
DEPTH = 4
N_COLLOCATION = 10000
ADAM_EPOCHS = 15000
ADAM_LR = 1.0e-3
LBFGS_MAX_ITER = 500
LBFGS_MAX_EVAL = 500
BOUNDARY_WEIGHT = 100.0
COARSE_POINTS = 101                   # Same coarse grid size as the LWM notebook.
PRINT_EVERY = 500
MAKE_PLOTS = True
SAVE_RESULTS = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
torch.set_default_dtype(DTYPE)
print("Device:", DEVICE, "| Precision:", DTYPE)


# 2. Exact solution and evaluation grids

def exact_solution(x, Pe):
    x = np.asarray(x, dtype=np.float64)
    return (1.0 - np.exp(Pe * (x - 1.0))) / (1.0 - np.exp(-Pe))


def make_dense_grid():
    uniform_part = np.linspace(0.0, 1.0, 2001)
    layer_part = 1.0 - np.geomspace(1.0e-14, 1.0, 5000)
    return np.unique(np.concatenate([uniform_part, layer_part, [0.0, 1.0]]))


# 3. Neural network

class PINN(nn.Module):
    def __init__(self, hard_bc=False):
        super().__init__()
        self.hard_bc = hard_bc
        layers = [nn.Linear(1, WIDTH), nn.Tanh()]
        for _ in range(DEPTH - 1):
            layers.extend([nn.Linear(WIDTH, WIDTH), nn.Tanh()])
        layers.append(nn.Linear(WIDTH, 1))
        self.net = nn.Sequential(*layers)

        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        u = self.net(x)
        if self.hard_bc:
            u = 1.0 - x + x * (1.0 - x) * u
        return u


# 4. PINN loss

def compute_losses(model, x, Pe):
    u = model(x)
    u_x = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u), create_graph=True
    )[0]
    u_xx = torch.autograd.grad(
        u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True
    )[0]
    residual_loss = torch.mean((u_xx - Pe * u_x) ** 2)

    if model.hard_bc:
        # Boundary values are built into the model, so train only the residual.
        zero = torch.zeros((), device=DEVICE, dtype=DTYPE)
        return residual_loss, residual_loss, zero

    x0 = torch.tensor([[0.0]], device=DEVICE, dtype=DTYPE)
    x1 = torch.tensor([[1.0]], device=DEVICE, dtype=DTYPE)
    boundary_loss = torch.mean((model(x0) - 1.0) ** 2 + model(x1) ** 2)
    total_loss = residual_loss + BOUNDARY_WEIGHT * boundary_loss
    return total_loss, residual_loss, boundary_loss


# 5. Train one model: Adam followed by L-BFGS

def train_pinn(Pe, seed, hard_bc=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = PINN(hard_bc=hard_bc).to(device=DEVICE, dtype=DTYPE)
    x = torch.linspace(0.0, 1.0, N_COLLOCATION, device=DEVICE, dtype=DTYPE)
    x = x.reshape(-1, 1).requires_grad_(True)
    adam = torch.optim.Adam(model.parameters(), lr=ADAM_LR)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    # Stage 1: Adam updates the network weights once per epoch.
    for epoch in range(1, ADAM_EPOCHS + 1):
        adam.zero_grad()
        x.grad = None
        total, residual, boundary = compute_losses(model, x, Pe)
        total.backward()
        adam.step()
        if epoch % PRINT_EVERY == 0 or epoch == ADAM_EPOCHS:
            print(f"Adam {epoch:6d} | total={total.item():.6e} | "
                  f"residual={residual.item():.6e} | BC={boundary.item():.6e}")

    # Stage 2: L-BFGS may evaluate the loss several times per iteration.
    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=1.0,
        max_iter=LBFGS_MAX_ITER, max_eval=LBFGS_MAX_EVAL,
        tolerance_grad=1.0e-12, tolerance_change=1.0e-12,
        history_size=100, line_search_fn="strong_wolfe"
    )

    def closure():
        # L-BFGS calls this function whenever it needs a fresh loss/gradient.
        lbfgs.zero_grad()
        x.grad = None
        total, _, _ = compute_losses(model, x, Pe)
        total.backward()
        return total

    print("Starting L-BFGS...")
    lbfgs.step(closure)
    total, residual, boundary = compute_losses(model, x, Pe)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    training_time = time.perf_counter() - start
    if not np.isfinite(total.item()):
        raise FloatingPointError("Training produced a non-finite loss.")

    state = lbfgs.state[next(model.parameters())]
    info = {
        "total_loss": total.item(), "residual_loss": residual.item(),
        "boundary_loss": boundary.item(), "training_time": training_time,
        "n_parameters": sum(p.numel() for p in model.parameters()),
        "lbfgs_iterations": int(state["n_iter"]),
        "lbfgs_evaluations": int(state["func_evals"])
    }
    return model, info


# 6. Evaluate the trained model

def predict(model, x):
    x_tensor = torch.tensor(np.asarray(x).reshape(-1, 1), device=DEVICE, dtype=DTYPE)
    model.eval()
    with torch.no_grad():
        return model(x_tensor).cpu().numpy().reshape(-1)


def evaluate_model(model, Pe):
    x = make_dense_grid()
    numerical = predict(model, x)
    exact = exact_solution(x, Pe)
    error = np.abs(numerical - exact)
    if not np.all(np.isfinite(error)):
        raise FloatingPointError("Evaluation produced non-finite values.")

    x_coarse = np.linspace(0.0, 1.0, COARSE_POINTS)
    coarse_error = np.abs(predict(model, x_coarse) - exact_solution(x_coarse, Pe))
    return {
        "x_dense": x, "u_dense": numerical, "u_exact": exact, "abs_error": error,
        "max_error": float(np.max(error)),
        "mean_error": float(np.mean(error)),
        "rms_error": float(np.sqrt(np.mean(error ** 2))),
        "l2_error": float(np.sqrt(np.trapezoid(error ** 2, x))),
        "left_boundary_error": abs(float(predict(model, [0.0])[0]) - 1.0),
        "right_boundary_error": abs(float(predict(model, [1.0])[0])),
        "coarse_max_error": float(np.max(coarse_error))
    }


# 7. Saving and plotting

def save_csv(filename, rows, header):
    if SAVE_RESULTS:
        np.savetxt(os.path.join(OUTPUT_DIR, filename), rows,
                   delimiter=",", header=header, comments="", fmt="%.17e")


def save_summary(filename, rows):
    if SAVE_RESULTS:
        with open(os.path.join(OUTPUT_DIR, filename), "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


def finish_plot(filename):
    plt.tight_layout()
    if SAVE_RESULTS:
        plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_comparison(Pe, soft, hard):
    x = soft["x_dense"]
    plt.figure(figsize=(7, 4.5))
    plt.plot(x, soft["u_exact"], "--", label="Exact")
    plt.plot(x, soft["u_dense"], label="Soft BC")
    plt.plot(x, hard["u_dense"], label="Hard BC")
    plt.xlabel("x")
    plt.ylabel("u(x)")
    plt.title(f"Pe={Pe:g}, seed={HARD_BC_SEED}")
    plt.legend()
    finish_plot(f"comparison_pe{int(Pe)}.png")

    xi = Pe * (1.0 - x)
    mask = xi <= 10.0
    order = np.argsort(xi[mask])
    plt.figure(figsize=(7, 4.5))
    plt.plot(xi[mask][order], soft["u_exact"][mask][order], "--", label="Exact")
    plt.plot(xi[mask][order], soft["u_dense"][mask][order], label="Soft BC")
    plt.plot(xi[mask][order], hard["u_dense"][mask][order], label="Hard BC")
    plt.xlabel(r"$\xi=Pe(1-x)$")
    plt.ylabel("u")
    plt.title(f"Boundary layer, Pe={Pe:g}, seed={HARD_BC_SEED}")
    plt.legend()
    finish_plot(f"stretched_pe{int(Pe)}.png")


# 8. Run one complete case

def run_case(Pe, seed, hard_bc=False):
    method = "hard" if hard_bc else "soft"
    print(f"\n{method.upper()} PINN: Pe={Pe:g}, seed={seed}")
    model, training = train_pinn(Pe, seed, hard_bc=hard_bc)
    info = evaluate_model(model, Pe)
    row = {
        "Pe": Pe, "nu": 1.0 / Pe, "seed": seed, "method": method,
        "total_loss": training["total_loss"],
        "residual_loss": training["residual_loss"],
        "boundary_loss": training["boundary_loss"],
        "E_inf": info["max_error"], "E2": info["l2_error"],
        "mean_error": info["mean_error"], "rms_error": info["rms_error"],
        "left_BE": info["left_boundary_error"], "right_BE": info["right_boundary_error"],
        "coarse_points": COARSE_POINTS, "coarse_E_inf": info["coarse_max_error"],
        "dense_points": len(info["x_dense"]),
        "training_time": training["training_time"], "parameters": training["n_parameters"],
        "lbfgs_iterations": training["lbfgs_iterations"],
        "lbfgs_evaluations": training["lbfgs_evaluations"]
    }
    if hard_bc:
        row["boundary_loss"] = row["left_BE"] ** 2 + row["right_BE"] ** 2
    # Save curves only for the soft/hard pair plotted below.
    if hard_bc or seed == HARD_BC_SEED:
        tag = f"{method}_pe{int(Pe)}_seed{seed}"
        dense_rows = np.column_stack([info["x_dense"], info["u_dense"],
                                      info["u_exact"], info["abs_error"]])
        save_csv(f"{tag}_dense.csv", dense_rows, "x,u_pinn,u_exact,absolute_error")
    print(f"E_inf={row['E_inf']:.12e} | E2={row['E2']:.12e} | "
          f"time={row['training_time']:.2f} s")
    print(f"Boundary errors: left={row['left_BE']:.12e}, right={row['right_BE']:.12e}")
    return row, info


# 9. Reproducibility information

def save_run_information():
    lines = [
        f"Python: {sys.version.split()[0]}; PyTorch: {torch.__version__}; NumPy: {np.__version__}",
        f"Platform: {platform.platform()}; CPU: {platform.processor() or 'not reported'}",
        f"Device: {DEVICE}; precision: {DTYPE}",
        f"Pe cases: {PE_CASES}; seeds: {SEEDS}",
        f"Hard-boundary seed: {HARD_BC_SEED}",
        f"Network: {DEPTH} hidden layers, width {WIDTH}, tanh, Xavier-normal weights",
        f"Training grid: {N_COLLOCATION} equally spaced points including both endpoints",
        f"Adam: {ADAM_EPOCHS} updates; lr={ADAM_LR}",
        f"L-BFGS: lr=1; max_iter={LBFGS_MAX_ITER}; max_eval={LBFGS_MAX_EVAL}",
        "L-BFGS: tolerance_grad=1e-12; tolerance_change=1e-12; history_size=100; strong_wolfe",
        f"Soft boundary weight: {BOUNDARY_WEIGHT}; actual optimizer counts are in the CSVs",
        "Dense grid: unique union of linspace(0,1,2001), 1-geomspace(1e-14,1,5000), and [0,1]",
        f"Dense points: {len(make_dense_grid())}; coarse grid: {COARSE_POINTS} uniform points",
        "E2 uses trapezoidal integration; mean/RMS are unweighted dense-grid statistics",
        "Timing includes optimization, progress printing and final training-loss evaluation",
        "Network/grid/Adam setup, error evaluation, plotting and saving are outside the timer",
        "Seeds are reset for each model; different hardware/software may change results",
    ]
    if DEVICE.type == "cuda":
        lines.append(f"GPU: {torch.cuda.get_device_name(0)}; CUDA: {torch.version.cuda}")
    if SAVE_RESULTS:
        with open(os.path.join(OUTPUT_DIR, "run_information.txt"), "w") as file:
            file.write("\n".join(lines) + "\n")


# 10. Main program

def main():
    if len(SEEDS) < 5 or len(SEEDS) != len(set(SEEDS)) or HARD_BC_SEED not in SEEDS:
        raise ValueError("Use at least five distinct seeds, including HARD_BC_SEED.")
    if SAVE_RESULTS:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    save_run_information()
    soft_rows, hard_rows, statistics, comparisons = [], [], [], []

    for Pe in PE_CASES:
        # Five independent standard PINNs for this Pe.
        case_rows = []
        for seed in SEEDS:
            row, info = run_case(Pe, seed, hard_bc=False)
            case_rows.append(row)
            soft_rows.append(row)
            save_summary("soft_pinn_all_seeds.csv", soft_rows)
            if seed == HARD_BC_SEED:
                matched_soft_row, matched_soft_info = row, info

        # Calculate mean and sample SD from the full-precision errors.
        e_inf = np.array([row["E_inf"] for row in case_rows])
        e2 = np.array([row["E2"] for row in case_rows])
        statistics.append({
            "Pe": Pe, "n_seeds": len(SEEDS),
            "mean_E_inf": np.mean(e_inf), "std_E_inf": np.std(e_inf, ddof=1),
            "min_E_inf": np.min(e_inf), "max_E_inf": np.max(e_inf),
            "mean_E2": np.mean(e2), "std_E2": np.std(e2, ddof=1)
        })
        save_summary("soft_pinn_statistics.csv", statistics)
        print(f"Soft PINN: mean(E_inf)={np.mean(e_inf):.12e}, "
              f"sample SD={np.std(e_inf, ddof=1):.12e}")

        # One hard-boundary PINN, paired with soft seed 1234 by default.
        hard_row, hard_info = run_case(Pe, HARD_BC_SEED, hard_bc=True)
        hard_rows.append(hard_row)
        save_summary("hard_pinn_results.csv", hard_rows)
        comparisons.append({
            "Pe": Pe, "seed": HARD_BC_SEED,
            "soft_E_inf": matched_soft_row["E_inf"], "hard_E_inf": hard_row["E_inf"],
            "soft_E2": matched_soft_row["E2"], "hard_E2": hard_row["E2"],
            "soft_left_BE": matched_soft_row["left_BE"], "soft_right_BE": matched_soft_row["right_BE"],
            "hard_left_BE": hard_row["left_BE"], "hard_right_BE": hard_row["right_BE"],
            "soft_time": matched_soft_row["training_time"], "hard_time": hard_row["training_time"]
        })
        save_summary("soft_vs_hard_comparison.csv", comparisons)
        if MAKE_PLOTS:
            plot_comparison(Pe, matched_soft_info, hard_info)

    print("\nPe        Soft E_inf         Hard E_inf")
    for row in comparisons:
        print(f"{row['Pe']:6g}  {row['soft_E_inf']:.12e}  {row['hard_E_inf']:.12e}")
    print(f"\nCompleted. Output folder: {OUTPUT_DIR}")
    return soft_rows, hard_rows, statistics, comparisons


if __name__ == "__main__":
    SOFT_RESULTS, HARD_RESULTS, STATISTICS, COMPARISONS = main()
